In [48]:
import os, random, math
from pathlib import Path
from collections import Counter
import cv2

import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
import itertools
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from sklearn.model_selection import train_test_split
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [49]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

DATASET_PATH = r"C:\Users\zhylk\Unicourses\ML\ASL_Alphabet_Dataset\asl_alphabet_train"
IMG_SIZE = 96
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),           # converts to channels-first + scales to 0-1
])

full_dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
classes = full_dataset.classes
NUM_CLASSES = len(classes)

print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Total images: {len(full_dataset)}")

# Split: 70% train, 15% val, 15% test
train_size = int(0.70 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Classes (29): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Total images: 223074
Train: 156151, Val: 33461, Test: 33462


In [50]:
import torch
import torch.nn as nn

NUM_CLASSES = 29
IMG_SIZE = 96

class ASLModel(nn.Module):
    def __init__(self):
        super(ASLModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 10 * 10, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ASLModel().to(device)
print(f"Using device: {device}")
print(model)

Using device: cuda
ASLModel(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=12800, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=29, bias=True)
  )
)


In [51]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # same as SparseCategoricalCrossentropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [52]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[]

In [53]:
import torch
print(f"Device: {device}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"Model on GPU: {next(model.parameters()).is_cuda}")

Device: cuda
torch.cuda.is_available(): True
Model on GPU: True


In [ ]:
# Training Loop
EPOCHS = 8

history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

for epoch in range(EPOCHS):
    # Train 
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(y_batch)
        train_correct += (outputs.argmax(1) == y_batch).sum().item()
        train_total += len(y_batch)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * len(y_batch)
            val_correct += (outputs.argmax(1) == y_batch).sum().item()
            val_total += len(y_batch)

    history['loss'].append(train_loss / train_total)
    history['accuracy'].append(train_correct / train_total)
    history['val_loss'].append(val_loss / val_total)
    history['val_accuracy'].append(val_correct / val_total)

    print(f"Epoch {epoch+1}/{EPOCHS} — "
          f"loss: {history['loss'][-1]:.4f}, acc: {history['accuracy'][-1]:.4f} — "
          f"val_loss: {history['val_loss'][-1]:.4f}, val_acc: {history['val_accuracy'][-1]:.4f}")

In [ ]:
print(classes)
print(len(classes))

['asl_alphabet_test', 'asl_alphabet_train']
2


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_title('Model Loss over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"Final Train Accuracy : {history.history['accuracy'][-1]:.4f}")
print(f"Final Val Accuracy   : {history.history['val_accuracy'][-1]:.4f}")
print(f"Final Train Loss     : {history.history['loss'][-1]:.4f}")
print(f"Final Val Loss       : {history.history['val_loss'][-1]:.4f}")

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print("TEST SET EVALUATION")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_accuracy:.4f}")
print(f"  Macro Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"  Macro Recall   : {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"  Macro F1-Score : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"  Weighted F1    : {f1_score(y_true, y_pred, average='weighted'):.4f}")

In [ ]:
print("\nClassification Report (per-class breakdown):\n")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=classes, yticklabels=classes,
    linewidths=0.5, linecolor='gray', ax=ax
)
ax.set_title('Confusion Matrix — ASL Alphabet', fontsize=16)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=classes, yticklabels=classes,
    linewidths=0.5, linecolor='gray', ax=ax,
    vmin=0, vmax=1
)
ax.set_title('Normalized Confusion Matrix (Recall per class)', fontsize=16)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#e74c3c' if acc < 0.8 else '#2ecc71' if acc >= 0.95 else '#f39c12'
          for acc in per_class_acc]
bars = ax.bar(classes, per_class_acc, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=0.9, color='blue', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_title('Per-Class Accuracy', fontsize=14)
ax.set_xlabel('ASL Sign')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nClasses below 90% accuracy:")
for cls, acc in zip(classes, per_class_acc):
    if acc < 0.90:
        print(f"  {cls:>8s}: {acc:.4f}")

In [ ]:
# ROC Curves (One-vs-Rest)
y_test_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= NUM_CLASSES
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(10, 8))
for i in range(NUM_CLASSES):
    ax.plot(fpr[i], tpr[i], alpha=0.3, linewidth=1)

ax.plot(all_fpr, mean_tpr, color='navy', linewidth=2.5,
        label=f'Macro-avg ROC (AUC = {macro_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves - One-vs-Rest (all 29 classes)', fontsize=14)
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMacro-average AUC: {macro_auc:.4f}")
print(f"\nPer-class AUC (sorted worst to best):")
sorted_auc = sorted(roc_auc.items(), key=lambda x: x[1])
for idx, auc_val in sorted_auc[:10]:
    print(f"  {classes[idx]:>8s}: {auc_val:.4f}")

In [ ]:
precision_dict, recall_dict, avg_prec = {}, {}, {}
for i in range(NUM_CLASSES):
    precision_dict[i], recall_dict[i], _ = precision_recall_curve(
        y_test_bin[:, i], y_pred_probs[:, i]
    )
    avg_prec[i] = average_precision_score(y_test_bin[:, i], y_pred_probs[:, i])

macro_ap = np.mean(list(avg_prec.values()))

fig, ax = plt.subplots(figsize=(10, 8))
for i in range(NUM_CLASSES):
    ax.plot(recall_dict[i], precision_dict[i], alpha=0.3, linewidth=1)

ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title(f'Precision-Recall Curves — Macro AP = {macro_ap:.4f}', fontsize=14)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Macro Average Precision: {macro_ap:.4f}")

In [ ]:
misclass_pairs = {}
for t, p in zip(y_true, y_pred):
    if t != p:
        pair = (classes[t], classes[p])
        misclass_pairs[pair] = misclass_pairs.get(pair, 0) + 1

sorted_pairs = sorted(misclass_pairs.items(), key=lambda x: x[1], reverse=True)

print("Top 15 Misclassification Pairs:")
print(f"{'True':>10s} to {'Predicted':<10s}  Count")
print("-" * 35)
for (true_cls, pred_cls), count in sorted_pairs[:15]:
    print(f"{true_cls:>10s} → {pred_cls:<10s}  {count}")

# Visualise top 10 pairs
if len(sorted_pairs) >= 5:
    top_n = min(10, len(sorted_pairs))
    pair_labels = [f"{t}→{p}" for (t, p), _ in sorted_pairs[:top_n]]
    pair_counts = [c for _, c in sorted_pairs[:top_n]]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(pair_labels[::-1], pair_counts[::-1], color='#e74c3c', edgecolor='black')
    ax.set_xlabel('Count')
    ax.set_title(f'Top {top_n} Misclassification Pairs', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
misclassified_idx = np.where(y_pred != y_true)[0]

if len(misclassified_idx) > 0:
    n_show = min(16, len(misclassified_idx))
    sample_idx = np.random.choice(misclassified_idx, n_show, replace=False)

    cols = 4
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.5 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(sample_idx):
        axes[i].imshow(X_test[idx])
        axes[i].set_title(
            f"True: {classes[y_true[idx]]}\nPred: {classes[y_pred[idx]]}",
            fontsize=9, color='red'
        )
        axes[i].axis('off')

    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    plt.suptitle('Misclassified Samples', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified samples found — perfect test accuracy!")

print(f"\nTotal misclassified: {len(misclassified_idx)} / {len(y_true)} "
      f"({len(misclassified_idx)/len(y_true)*100:.1f}%)")

In [ ]:
max_probs = np.max(y_pred_probs, axis=1)
correct_mask = (y_pred == y_true)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_probs[correct_mask], bins=50, alpha=0.7, label='Correct', color='#2ecc71', edgecolor='black')
ax.hist(max_probs[~correct_mask], bins=50, alpha=0.7, label='Incorrect', color='#e74c3c', edgecolor='black')
ax.set_xlabel('Max Predicted Probability', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Prediction Confidence Distribution', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean confidence (correct)  : {max_probs[correct_mask].mean():.4f}")
if (~correct_mask).sum() > 0:
    print(f"Mean confidence (incorrect): {max_probs[~correct_mask].mean():.4f}")